# 02 光谱分析 Pipeline

这个 notebook 用一个可复现 pipeline 替代原来的多个探索 notebook。它从 `data/` 读取定标后的一维 FITS 光谱，测量适合稀疏光谱样本的保守诊断量，把 CSV 表写到 `output/analysis_pipeline/`，并生成可直接放进报告的图。

根据 `paper/sparse-multi-epoch-sn-spectra/` 的文献调研：每个目标只有 1-3 条光谱时，稳妥结论应集中在类型/子型检查、光谱相位、速度、pEW/FWHM、宿主污染和公开样本比较上。TARDIS 只作为解释辅助，不作为主要证据。

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.spectral_pipeline import build_all

ANALYSIS_DIR = PROJECT_ROOT / "output" / "analysis_pipeline"
FIG_DIR = ANALYSIS_DIR / "figures"

## 运行或刷新分析产物

In [ ]:
RUN_PIPELINE = True

if RUN_PIPELINE:
    paths = build_all(PROJECT_ROOT)
    print(f"已更新 {ANALYSIS_DIR}")
    for item in paths.get("figures", []):
        print(item)
else:
    print("使用现有 output/analysis_pipeline 产物。")

## 目标层面的科学状态

In [ ]:
target_status = pd.read_csv(ANALYSIS_DIR / "target_status.csv")
display(target_status)

## 带质检标记的谱线诊断

`qc_flag=adopt` 表示自动测量通过保守检查，可以用于第一版图表。`qc_flag=check` 表示在写入科学结论前必须人工看图确认。这样可以避免把噪声谷、宽线混合或次要谱线误读成可靠物理量。

In [ ]:
line_qc = pd.read_csv(ANALYSIS_DIR / "line_diagnostics_qc.csv")
display(line_qc[["target", "date_obs", "phase_days", "type", "line", "velocity_kms", "pEW_A", "FWHM_A", "qc_flag", "qc_note"]])

## 宿主星系/环境诊断

In [ ]:
host_summary = pd.read_csv(ANALYSIS_DIR / "host_environment_summary.csv")
host_lines = pd.read_csv(ANALYSIS_DIR / "host_environment_lines.csv")
display(host_summary)
display(host_lines[host_lines["status"].eq("detected")].head(30))

## 报告可用图表

In [ ]:
for fig in [
    "target_status_table.png",
    "line_velocity_evolution.png",
    "pew_evolution.png",
    "blackbody_temperature.png",
    "host_line_detections.png",
]:
    path = FIG_DIR / fig
    if path.exists():
        display(Markdown(f"### {fig}"))
        display(Image(filename=str(path)))

## 多历元光谱序列

In [ ]:
for path in sorted(FIG_DIR.glob("spectral_sequence_*.png")):
    display(Markdown(f"### {path.stem.replace('_', ' ')}"))
    display(Image(filename=str(path)))